# Evaluation file


### Uses the exported results from the artifacts folder
to avoid rerunning entire notebooks

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

ART = Path("artifacts")

data = pd.read_pickle(ART / "preprocessed_data.pkl")

tfidf_candidates = pd.read_csv(
    ART / "tfidf_candidates.csv"
)

candidate_scores = pd.read_csv(
    ART / "candidate_scores.csv"
)

candidate_post_scores = pd.read_csv(
    ART / "candidate_post_scores.csv"
)

candidate_metadata = pd.read_csv(
    ART / "candidate_metadata.csv"
)

anchor_metadata = pd.read_csv(
    ART / "anchor_metadata.csv"
)

### Define known expressions

cf Table IV of Kikkisetti et al. (2024)


In [11]:
known_terms = [
    "white genocide",
    "deep state",
    "interest groups",
    "kazarian mafia",
]

new_terms = [
    "fema camps",
    "klaus schwab",
    "central bank",
    "national socialist",
]

false_positives = [
    "plain sight",
    "german people",
    "new york city",
    "big part",
]

gold = pd.DataFrame({
    "term": known_terms + new_terms + false_positives,
    "label": (
        ["known"] * len(known_terms)
        + ["new"] * len(new_terms)
        + ["negative"] * len(false_positives)
    ),
})

display(gold)

,term,label
0,white genocide,known
1,deep state,known
2,interest groups,known
3,kazarian mafia,known
4,fema camps,new
5,klaus schwab,new
6,central bank,new
7,national socialist,new
8,plain sight,negative
9,german people,negative


### recover the vocabulary at each stage

In [12]:
def gram_to_string(gram):
    if isinstance(gram, str):
        return gram.casefold()
    return " ".join(gram).casefold()


def extract_vocabulary(columns):
    vocabulary = set()

    for column in columns:
        for grams in data[column]:
            for gram in grams:
                vocabulary.add(gram_to_string(gram))

    return vocabulary


vocab_before_pos = extract_vocabulary([
    "Unigrams",
    "Bigrams",
    "Trigrams",
])

vocab_after_pos = extract_vocabulary([
    "Unigrams Filtered",
    "Bigrams Filtered",
    "Trigrams Filtered",
])

vocab_after_tfidf = set(
    tfidf_candidates.loc[
        tfidf_candidates["selected"],
        "candidate",
    ]
)

vocab_scored = set(candidate_scores["candidate"])

print("Before POS:", len(vocab_before_pos))
print("After POS:", len(vocab_after_pos))
print("After TF-IDF:", len(vocab_after_tfidf))
print("Scored candidates:", len(vocab_scored))

Before POS: 80997
After POS: 21851
After TF-IDF: 8634
Scored candidates: 8634


#### Check if expression exists in the raw corpus

In [13]:
def count_posts_with_term(term):
    # Match a complete expression, allowing varying whitespace.
    pattern = re.compile(
        r"(?<!\w)"
        + r"\s+".join(re.escape(w) for w in term.split())
        + r"(?!\w)",
        flags=re.IGNORECASE,
    )

    return sum(
        bool(pattern.search(text))
        for text in data["Text for BERT"]
        if isinstance(text, str)
    )


gold["raw_posts"] = gold["term"].apply(
    count_posts_with_term
)

gold["in_raw"] = gold["raw_posts"] > 0

display(gold)

,term,label,raw_posts,in_raw
0,white genocide,known,5,True
1,deep state,known,12,True
2,interest groups,known,3,True
3,kazarian mafia,known,0,False
4,fema camps,new,2,True
5,klaus schwab,new,9,True
6,central bank,new,3,True
7,national socialist,new,3,True
8,plain sight,negative,4,True
9,german people,negative,5,True


### trace every expression through the pipeline

In [14]:
gold["generated"] = gold["term"].isin(
    vocab_before_pos
)

gold["after_pos"] = gold["term"].isin(
    vocab_after_pos
)

gold["after_tfidf"] = gold["term"].isin(
    vocab_after_tfidf
)

gold["scored"] = gold["term"].isin(
    vocab_scored
)

# Global HMApp ranking.
ranking = (
    candidate_scores
    .sort_values("relevance", ascending=False)
    .reset_index(drop=True)
)

ranking["rank"] = np.arange(1, len(ranking) + 1)

rank_lookup = ranking.set_index("candidate")["rank"]
score_lookup = ranking.set_index("candidate")["relevance"]

gold["rank"] = gold["term"].map(rank_lookup)
gold["relevance"] = gold["term"].map(score_lookup)

# Published HMApp relevance cutoff.
threshold = candidate_scores["relevance"].mean()

gold["passes_hmapp"] = (
    gold["relevance"] > threshold
)

display(gold)

,term,label,raw_posts,in_raw,generated,after_pos,after_tfidf,scored,rank,relevance,passes_hmapp
0,white genocide,known,5,True,True,True,True,True,791.0,0.617182,True
1,deep state,known,12,True,True,True,True,True,2967.0,0.533629,True
2,interest groups,known,3,True,True,True,True,True,4966.0,0.483584,False
3,kazarian mafia,known,0,False,False,False,False,False,NaN,NaN,False
4,fema camps,new,2,True,True,True,True,True,2881.0,0.536202,True
5,klaus schwab,new,9,True,True,True,True,True,4715.0,0.489486,False
6,central bank,new,3,True,True,True,True,True,6504.0,0.442916,False
7,national socialist,new,3,True,True,True,False,False,NaN,NaN,False
8,plain sight,negative,4,True,True,True,True,True,6210.0,0.451854,False
9,german people,negative,5,True,True,True,True,True,1480.0,0.582787,True


### identify the failure stage

In [15]:
def failure_stage(row):
    if not row["in_raw"]:
        return "Absent from raw corpus"

    if not row["generated"]:
        return "N-gram generation / preprocessing"

    if not row["after_pos"]:
        return "POS filtering"

    if not row["after_tfidf"]:
        return "TF-IDF filtering"

    if not row["scored"]:
        return "Occurrence matching / embedding"

    if not row["passes_hmapp"]:
        return "Semantic relevance cutoff"

    return "Retained by HMApp"


gold["outcome"] = gold.apply(
    failure_stage,
    axis=1,
)

display(
    gold[[
        "term",
        "label",
        "raw_posts",
        "outcome",
        "rank",
        "relevance",
    ]]
)

,term,label,raw_posts,outcome,rank,relevance
0,white genocide,known,5,Retained by HMApp,791.0,0.617182
1,deep state,known,12,Retained by HMApp,2967.0,0.533629
2,interest groups,known,3,Semantic relevance cutoff,4966.0,0.483584
3,kazarian mafia,known,0,Absent from raw corpus,NaN,NaN
4,fema camps,new,2,Retained by HMApp,2881.0,0.536202
5,klaus schwab,new,9,Semantic relevance cutoff,4715.0,0.489486
6,central bank,new,3,Semantic relevance cutoff,6504.0,0.442916
7,national socialist,new,3,TF-IDF filtering,NaN,NaN
8,plain sight,negative,4,Semantic relevance cutoff,6210.0,0.451854
9,german people,negative,5,Retained by HMApp,1480.0,0.582787


## Eval metrics

### Candidate-stage recall

In [16]:
positives = gold.loc[
    gold["label"].isin(["known", "new"])
    & gold["in_raw"]
]

for stage in [
    "generated",
    "after_pos",
    "after_tfidf",
    "scored",
    "passes_hmapp",
]:
    rate = positives[stage].mean()
    print(f"{stage}: {rate:.1%}")

generated: 100.0%
after_pos: 100.0%
after_tfidf: 85.7%
scored: 85.7%
passes_hmapp: 42.9%


### Ranking quality

In [17]:
display(
    gold.loc[
        gold["label"].isin(["known", "new"]),
        ["term", "rank", "relevance"]
    ].sort_values("rank")
)

,term,rank,relevance
0,white genocide,791.0,0.617182
4,fema camps,2881.0,0.536202
1,deep state,2967.0,0.533629
5,klaus schwab,4715.0,0.489486
2,interest groups,4966.0,0.483584
6,central bank,6504.0,0.442916
3,kazarian mafia,NaN,NaN
7,national socialist,NaN,NaN
